<a href="https://colab.research.google.com/github/BrajanNieto/DeepLearning/blob/main/Grupo4_Laboratorio03_Whisper_Qwen_XTTSv2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Deep Learning - Multimodal Conversational AI: Whisper, Qwen & XTTS v2**

---

This repository contains the implementation of a **Real-Time Multimodal Voice Chatbot**. The project integrates Speech-to-Text (ASR), Large Language Models (LLM), and Zero-Shot Voice Cloning to create a seamless conversational experience. By using quantized models and GPU acceleration, it achieves low-latency responses with personalized vocal identities.

### Key Capabilities:
* **Remote Voice Sync:** Automated retrieval of pre-extracted vocal embeddings (`.pth`) from GitHub to the local environment.
* **Intelligent Reasoning:** Integration of **Qwen 2.5 1.5B (Quantized GGUF)** for fast, natural, and context-aware Spanish responses.
* **Instant Transcription:** Real-time audio-to-text conversion using **OpenAI Whisper** with CUDA optimization.
* **Zero-Shot Synthesis:** High-fidelity voice cloning via **XTTS v2**, allowing the assistant to speak with any stored identity.
* **Interactive UI:** A fully functional **Gradio** interface for microphone input, voice selection, and real-time audio playback.


**Authors:**

Lopez Medina, Sebastian  
[sebastian.lopez@utec.edu.pe](mailto:sebastian.lopez@utec.edu.pe)

Nieto Espinoza, Brajan  
[brajan.nieto@utec.edu.pe](mailto:brajan.nieto@utec.edu.pe)

Tapia Chasquibol, Mateo  
[mateo.tapia@utec.edu.pe](mailto:mateo.tapia@utec.edu.pe)

---

# 0. Configuración de Entorno, Librerías y Dataser

* **Librerías:** Importación de PyTorch, herramientas de manejo de datos y visualización.
* **Reproducibilidad:** Configuración de `SEED` y detección de aceleración por hardware (CUDA).
* **Fuentes de Datos:** Definición de diccionarios de URLs y constantes para la selección de variables del dataset.

In [1]:
!pip install -q coqui-tts torch torchaudio
!pip install -q openai-whisper
!pip install -q llama-cpp-python
!pip install -q gradio
!apt-get -qq install ffmpeg
print(' Dependencias instaladas')

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 862.8/862.8 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 997.3/997.3 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 648.4/648.4 kB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 21.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.0/59.0 MB 21.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing b

In [2]:
import requests
import os
from google.colab import files
import torch

# 1. Voice Asset Synchronization: Remote Embedding Integration

Implementation of a **remote asset synchronization** layer that pulls pre-extracted vocal embeddings directly from the GitHub repository into the local runtime. This allows for the immediate use of previously cloned voices without requiring raw audio samples or re-extraction.

* **Automated Asset Discovery:** Leverages the GitHub API to scan the `cloned_voice/` directory, dynamically identifying and listing all available `.pth` vocal profiles for the current session.
* **Efficient Binary Transfer:** Downloads serialized PyTorch tensors via the GitHub raw content delivery network, ensuring high-speed access to the `gpt_cond_latent` and `speaker_embedding` data.
* **Integrity Validation:** Performs a post-download verification by loading each file with `torch.load`. It confirms the tensor shapes to ensure the vocal identity data is intact and ready for the synthesis engine.
* **Vocal Registry Mapping:** Populates a `voces_disponibles` dictionary that maps person identifiers to their local paths, creating an organized registry for streamlined multi-voice inference.

In [3]:
# ============================================================
# CONFIGURACIÓN - Tu repositorio
# ============================================================
GITHUB_USER = 'BrajanNieto'
GITHUB_REPO = 'DeepLearning'
GITHUB_BRANCH = 'main'
CLONED_FOLDER = 'cloned_voice'

API_URL = f'https://api.github.com/repos/{GITHUB_USER}/{GITHUB_REPO}/contents/{CLONED_FOLDER}?ref={GITHUB_BRANCH}'
RAW_BASE = f'https://raw.githubusercontent.com/{GITHUB_USER}/{GITHUB_REPO}/{GITHUB_BRANCH}/{CLONED_FOLDER}'

LOCAL_VOICES = '/content/voices'
os.makedirs(LOCAL_VOICES, exist_ok=True)

# Listar archivos .pth en cloned_voice/
print(f'Buscando voces en GitHub...')
response = requests.get(API_URL)
response.raise_for_status()
files = response.json()

pth_files = [f for f in files if f['name'].endswith('.pth')]
print(f'Voces encontradas: {len(pth_files)}')

# Descargar cada .pth
voces_disponibles = {}

for f in pth_files:
    nombre = f['name']
    persona = nombre.replace('_voice.pth', '')
    url = f'{RAW_BASE}/{nombre}'
    local_path = os.path.join(LOCAL_VOICES, nombre)

    print(f'\n  Descargando: {nombre}...')
    r = requests.get(url)
    r.raise_for_status()
    with open(local_path, 'wb') as out:
        out.write(r.content)

    # Verificar que se carga bien
    data = torch.load(local_path, map_location='cpu')
    voces_disponibles[persona] = local_path
    print(f'   {persona} | gpt_cond: {data["gpt_cond_latent"].shape} | speaker_emb: {data["speaker_embedding"].shape}')

print(f'\n{"="*50}')
print(f' {len(voces_disponibles)} voces listas para usar')
for nombre in voces_disponibles:
    print(f'  {nombre}')
print(f'{"="*50}')

Buscando voces en GitHub...
Voces encontradas: 3

  Descargando: BNE_Voice_voice.pth...
   BNE_Voice | gpt_cond: torch.Size([1, 32, 1024]) | speaker_emb: torch.Size([1, 512, 1])

  Descargando: MTC_Voice_voice.pth...
   MTC_Voice | gpt_cond: torch.Size([1, 32, 1024]) | speaker_emb: torch.Size([1, 512, 1])

  Descargando: SLM_Voice_voice.pth...
   SLM_Voice | gpt_cond: torch.Size([1, 32, 1024]) | speaker_emb: torch.Size([1, 512, 1])

 3 voces listas para usar
  BNE_Voice
  MTC_Voice
  SLM_Voice


# 2. LLM Acquisition: Quantized Model Deployment

Implementation of the **Large Language Model (LLM)** acquisition layer, focusing on the deployment of a quantized version of **Qwen 2.5 1.5B**. This component provides the cognitive core for generating text that will later be synthesized into voice, optimized for high-performance inference in resource-constrained environments.

* **Quantized Weight Retrieval:** Downloads the **GGUF (Q4_K_M)** version of Qwen 2.5 from Hugging Face. This 4-bit quantization reduces the model size to approximately 1 GB, maintaining a high balance between linguistic accuracy and memory efficiency.
* **Idempotent Storage Management:** Implements a conditional check to verify the local existence of the model before initiating the download. This prevents redundant bandwidth consumption and speeds up the initialization process in persistent runtimes.
* **Directory Infrastructure:** Automates the creation of the `/content/modelos` path, ensuring a structured storage environment for large binary assets (BLOBs).
* **Transfer Monitoring:** Utilizes `wget` with real-time progress tracking and post-download size validation to guarantee the integrity of the model weights before they are loaded into memory.

In [4]:
QWEN_DIR = '/content/modelos'
os.makedirs(QWEN_DIR, exist_ok=True)

QWEN_MODEL = os.path.join(QWEN_DIR, 'qwen2.5-1.5b-instruct-q4_k_m.gguf')

if not os.path.exists(QWEN_MODEL):
    print('Descargando Qwen 2.5 1.5B cuantizado Q4 (~1 GB)...')
    !wget -q --show-progress -O {QWEN_MODEL} \
        'https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct-GGUF/resolve/main/qwen2.5-1.5b-instruct-q4_k_m.gguf'
    print(f' Descargado: {os.path.getsize(QWEN_MODEL)/1e9:.2f} GB')
else:
    print(f' Modelo ya descargado: {QWEN_MODEL}')

⏳ Descargando Qwen 2.5 1.5B cuantizado Q4 (~1 GB)...
/content/modelos/qw 100%[===================>]   1.04G   195MB/s    in 5.3s    
✅ Descargado: 1.12 GB


# 3. Multimodal Engine Initialization: Whisper, Qwen, and XTTS v2

Implementation of the **central processing core**, integrating speech recognition (ASR), large language modeling (LLM), and speech synthesis (TTS) into a unified pipeline. This section orchestrates the loading of three distinct AI architectures, ensuring they are correctly mapped to the available hardware acceleration (GPU/CUDA).

* **Whisper ASR Integration:** Loads the **Whisper 'base'** model for efficient audio-to-text transcription. This serves as the primary input gateway, converting human speech into textual tokens for the LLM.
* **Quantized LLM Orchestration:** Initializes the **Qwen 2.5** model using `llama-cpp-python`. By configuring `n_gpu_layers=35`, the system offloads the Transformer layers to the GPU, significantly reducing token generation latency while maintaining a context window of **2048 tokens**.
* **Synthesizer Backbone Deployment:** Bootstraps the **XTTS v2** engine and moves its weights to the GPU device. Accessing the underlying `synthesizer.tts_model` allows for direct latent conditioning during the cloning process.
* **Proactive Voice Caching:** Implements a high-speed **memory cache (`voice_cache`)** for vocal embeddings. By pre-loading and moving `.pth` tensors to the GPU before inference starts, the system eliminates I/O bottlenecks and enables near-instantaneous voice switching during dialogue.
* **Resource Optimization:** Dynamically toggles between `cuda` and `cpu` across all models. This ensures the environment remains stable and prevents memory fragmentation by aligning all tensors and model weights to the same compute device.

In [5]:
import whisper
from TTS.api import TTS
from llama_cpp import Llama

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f' Dispositivo: {device}')

# --- Whisper ---
print('\n Cargando Whisper base...')
modelo_whisper = whisper.load_model('base', device=device)
print(' Whisper listo')

# --- Qwen ---
print('\n Cargando Qwen cuantizado...')
modelo_qwen = Llama(
    model_path=QWEN_MODEL,
    n_ctx=2048,
    n_threads=4,
    n_gpu_layers=35 if device == 'cuda' else 0,
    verbose=False,
)
print(' Qwen listo')

# --- XTTS v2 ---
print('\n Cargando XTTS v2...')
modelo_tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2')
modelo_tts.to(device)
modelo_xtts = modelo_tts.synthesizer.tts_model
print(' XTTS v2 listo')

# --- Precargar embeddings en memoria ---
print('\n Cargando embeddings de voces en memoria...')
voice_cache = {}
for nombre, path in voces_disponibles.items():
    data = torch.load(path, map_location=device)
    voice_cache[nombre] = {
        'gpt_cond_latent': data['gpt_cond_latent'].to(device),
        'speaker_embedding': data['speaker_embedding'].to(device),
    }
    print(f'    {nombre}')

print(f'\n{"="*50}')
print(' ¡Todo cargado! Listo para conversar.')
print(f'{"="*50}')

 Dispositivo: cuda

 Cargando Whisper base...


100%|████████████████████████████████████████| 139M/139M [00:00<00:00, 219MiB/s]


 Whisper listo

 Cargando Qwen cuantizado...


llama_context: n_ctx_seq (2048) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


 Qwen listo

⏳ Cargando XTTS v2...
 > You must confirm the following:
 | > "I have purchased a commercial license from Coqui: licensing@coqui.ai"
 | > "Otherwise, I agree to the terms of the non-commercial CPML: https://coqui.ai/cpml" - [y/n]
 | | > y


100%|██████████| 1.87G/1.87G [00:37<00:00, 49.5MiB/s]
4.37kiB [00:00, 9.71MiB/s]
361kiB [00:00, 56.8MiB/s]
100%|██████████| 32.0/32.0 [00:00<00:00, 97.4kiB/s]
100%|██████████| 7.75M/7.75M [00:00<00:00, 80.9MiB/s]


 XTTS v2 listo

⏳ Cargando embeddings de voces en memoria...
    BNE_Voice
    MTC_Voice
    SLM_Voice

 ¡Todo cargado! Listo para conversar.


# 4. Multimodal Pipeline Orchestration: ASR-LLM-TTS Integration

Implementation of the **end-to-end conversational logic**, defining the functional flow that transforms raw audio input into a cloned vocal response. This section encapsulates the core intelligence of the system, coordinating the data handoff between the transcription, reasoning, and synthesis modules.

* **Speech-to-Text Transcription (Whisper):** Defines the `transcribir` function, which utilizes OpenAI's Whisper to convert audio signals into normalized text. By enabling `fp16` during CUDA execution, it optimizes inference speed without sacrificing word error rate (WER).
* **Contextual Reasoning & NLP (Qwen):** Implements `generar_respuesta` using a structured **ChatML prompt template**. It constrains the LLM to act as a "friendly assistant" with concise 2-3 sentence outputs, ensuring the subsequent voice synthesis remains fluid and conversational.
* **Vocal Cloning Synthesis (XTTS v2):** Encapsulates the `sintetizar_voz` logic, which retrieves pre-cached embeddings to condition the audio generation. It forces a **24,000 Hz** output sample rate, ensuring high-fidelity playback of the cloned identity.
* **Unified Pipeline Orchestration:** The `pipeline_completo` function serves as the master controller, managing the sequential flow from input audio to final WAV generation. It includes error handling for empty audio captures and provides a detailed log of the "User-AI" exchange.
* **Efficient Audio Handling:** Uses `torchaudio` for tensor-to-WAV conversion, managing the necessary dimension unsqueezing and sampling rate configurations required for standard media player compatibility.

In [6]:
import numpy as np
import tempfile
import torchaudio


def transcribir(audio_path):
    """Whisper: Audio → Texto"""
    resultado = modelo_whisper.transcribe(
        audio_path,
        language='es',
        fp16=(device == 'cuda'),
    )
    return resultado['text'].strip()


def generar_respuesta(texto_usuario):
    """Qwen: Texto → Respuesta en texto"""
    prompt = (
        '<|im_start|>system\n'
        'Eres un asistente amigable. Responde en español de forma concisa '
        'y natural, como si hablaras en una conversación. Máximo 2-3 oraciones.\n'
        '<|im_end|>\n'
        f'<|im_start|>user\n{texto_usuario}\n<|im_end|>\n'
        '<|im_start|>assistant\n'
    )
    respuesta = modelo_qwen(
        prompt,
        max_tokens=200,
        temperature=0.7,
        top_p=0.9,
        stop=['<|im_end|>'],
        echo=False,
    )
    return respuesta['choices'][0]['text'].strip()


def sintetizar_voz(texto, nombre_voz):
    """XTTS v2: Texto → Audio con voz clonada"""
    voz = voice_cache[nombre_voz]
    out = modelo_xtts.inference(
        text=texto,
        language='es',
        gpt_cond_latent=voz['gpt_cond_latent'],
        speaker_embedding=voz['speaker_embedding'],
        temperature=0.7,
    )
    output_path = '/content/respuesta.wav'
    torchaudio.save(output_path, torch.tensor(out['wav']).unsqueeze(0), 24000)
    return output_path


def pipeline_completo(audio_path, nombre_voz):
    """Pipeline completo: Audio → Whisper → Qwen → XTTS → Audio"""
    if audio_path is None:
        return None, ' No se recibió audio'

    # 1. Transcribir
    texto = transcribir(audio_path)
    if not texto:
        return None, ' No se detectó habla'

    # 2. Generar respuesta
    respuesta = generar_respuesta(texto)

    # 3. Sintetizar
    audio_salida = sintetizar_voz(respuesta, nombre_voz)

    log = f' Tú dijiste: {texto}\n\n Respuesta ({nombre_voz}): {respuesta}'
    return audio_salida, log


print(' Pipeline definido')

 Pipeline definido


# 5. User Interface: Gradio Multimodal Deployment

Implementation of a **web-based interactive interface** using the `Gradio` framework. This component abstracts the complex underlying AI pipeline into a user-friendly dashboard, enabling real-time voice-to-voice communication and providing a seamless "Human-in-the-Loop" experience.

* **Multimodal Input Management:** Configures the `gr.Audio` component to accept both live microphone streams and file uploads. This flexibility allows for immediate testing of the ASR (Whisper) module across different recording qualities.
* **Dynamic Identity Selection:** Populates a dropdown menu (`gr.Dropdown`) with the keys from the `voice_cache`. This enables the user to switch between different cloned vocal identities (extracted in earlier phases) on the fly without reloading the models.
* **Asynchronous Event Handling:** Maps the `btn.click` event to the `pipeline_completo` function. This orchestration handles the data flow from the input audio to the output synthesis, ensuring that the interface remains responsive during processing.
* **Integrated Playback & Logging:** Deploys a dedicated `gr.Audio` output with `autoplay=True`, allowing for instantaneous vocal response. Simultaneously, a `gr.Textbox` provides visual feedback of the transcription and LLM reasoning, ensuring transparency in the conversation.
* **Public Gateway & Debugging:** Utilizes `demo.launch(share=True)` to generate a temporary public URL via a Gradio tunnel. This allows for remote testing and collaboration, while the `debug=True` flag provides real-time stack traces for pipeline optimization.

In [7]:
import gradio as gr

nombres_voces = list(voice_cache.keys())

with gr.Blocks(title='Chatbot con Voz Clonada', theme=gr.themes.Soft()) as demo:
    gr.Markdown('#  Chatbot con Voz Clonada')
    gr.Markdown(
        '**Pipeline:** Micrófono → Whisper (transcribe) → '
        'Qwen 2.5 cuantizado (responde) → XTTS v2 (habla con voz clonada)'
    )

    with gr.Row():
        with gr.Column(scale=1):
            audio_input = gr.Audio(
                sources=['microphone', 'upload'],
                type='filepath',
                label=' Tu pregunta'
            )
            voz_dropdown = gr.Dropdown(
                choices=nombres_voces,
                value=nombres_voces[0],
                label=' ¿Con qué voz quieres que responda?'
            )
            btn = gr.Button(' Enviar', variant='primary', size='lg')

        with gr.Column(scale=1):
            audio_output = gr.Audio(
                label=' Respuesta con voz clonada',
                type='filepath',
                autoplay=True
            )
            log_output = gr.Textbox(
                label=' Transcripción y respuesta',
                lines=5
            )

    btn.click(
        fn=pipeline_completo,
        inputs=[audio_input, voz_dropdown],
        outputs=[audio_output, log_output]
    )

demo.launch(debug=True, share=True)

/tmp/ipykernel_3160/1343741428.py:5: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title='Chatbot con Voz Clonada', theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://1fc5aee3e081c60a11.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://1fc5aee3e081c60a11.gradio.live


# 6. Direct Prompt Inferencia: Manual Voice Synthesis

Implementation of a **text-driven inference bypass**, allowing for the direct generation of cloned speech without requiring audio input. This section is designed for rapid testing of the LLM (Qwen) and TTS (XTTS v2) integration, providing a streamlined path to verify response quality and vocal consistency.

* **Hardcoded Input Parameterization:** Features a dedicated configuration block for the `PREGUNTA` (Prompt) and `VOZ` (Identity) variables. This allows the user to manually override the conversational pipeline for targeted debugging or specific content generation.
* **Synchronous LLM Reasoning:** Calls the `generar_respuesta` function to process the manual text prompt. It leverages the pre-configured system instructions to ensure the output remains concise and optimized for the subsequent vocal synthesis.
* **Targeted Vocal Synthesis:** Executes the `sintetizar_voz` module using the selected identity from the memory cache. This verifies that the `gpt_cond_latent` and `speaker_embedding` are being correctly applied to the specific text generated by the LLM.
* **Instantaneous Audio Playback:** Utilizes `IPython.display.Audio` with `autoplay=True` for immediate acoustic feedback. This is essential for evaluating the "naturalness" of the cloned voice and checking for any potential artifacts in the generated waveform.
* **Detailed Execution Logging:** Provides clear terminal output of the AI's textual reasoning before the audio plays, ensuring full traceability of the data transformation from text to cloned speech.

In [8]:
from IPython.display import Audio, display

# ============================================================
# MODIFICA AQUÍ
# ============================================================
PREGUNTA = '¿Qué es la inteligencia artificial?'   # Tu pregunta
VOZ = nombres_voces[0]                              # Nombre de la voz
# ============================================================

print(f' Pregunta: {PREGUNTA}')
print(f' Voz: {VOZ}\n')

respuesta = generar_respuesta(PREGUNTA)
print(f' Qwen: {respuesta}\n')

audio_path = sintetizar_voz(respuesta, VOZ)
print(f' Reproduciendo con voz de {VOZ}:')
display(Audio(audio_path, autoplay=True))

 Pregunta: ¿Qué es la inteligencia artificial?
 Voz: BNE_Voice

 Qwen: La inteligencia artificial es el campo que estudia y desarrolla computadoras capaces de realizar tareas que normalmente requiere la inteligencia humana.

 Reproduciendo con voz de BNE_Voice:
